# 01 — Clean vs noisy traces:

To make noise look like real consumer GPS, a Gaussian jitter around the road centreline plus clean dropouts and a visible tunnel gap were used.

Three routes, one per noise level (σ = 15 / 25 / 40 m), each with ~10% dropouts and one ~60 s tunnel gap. 

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np

from src.graph import load_graph, as_routing_graph
from src.synthesize import corrupt, sample_route

M_PER_DEG = 111_320.0

nodes, edges = load_graph()
print(f"nodes={len(nodes)} edges={len(edges)} edge-km={edges['length_m'].sum()/1000:.1f}")
G = as_routing_graph(nodes, edges)

In [ ]:
routes = sample_route(G, n=3, seed=7)
for i, r in enumerate(routes):
    print(f"route {i}: length={r['length_m']:.0f} m  segments={len(r['segments'])}")

In [ ]:
SIGMAS = [15.0, 25.0, 40.0]
cases = []
for i, (route, sigma) in enumerate(zip(routes, SIGMAS)):
    fixes, truth = corrupt(route, sigma_m=sigma, dropout_p=0.1, gaps=1, seed=100 + i)
    cases.append((route, fixes, truth, sigma))
    kept = len(fixes["t"]) / len(truth["t"])
    print(f"sigma={sigma:.0f} m: truth={len(truth['t'])} fixes, kept={len(fixes['t'])} ({kept:.0%})")

In [ ]:
def to_xy(lat, lon, lat0):
    """Degrees -> local metres around lat0 (degrees)."""
    x = (np.asarray(lon) - np.mean(lon)) * M_PER_DEG * np.cos(np.radians(lat0))
    y = (np.asarray(lat) - np.mean(lat)) * M_PER_DEG
    return x, y


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharex=False, sharey=False)
for ax, (route, fixes, truth, sigma) in zip(axes, cases):
    lat0 = float(np.mean(truth["lat"]))
    tx, ty = to_xy(truth["lat"], truth["lon"], lat0)
    fx, fy = to_xy(fixes["lat"], fixes["lon"], lat0)
    ax.plot(tx, ty, color="black", lw=1.6, label="clean (ground truth)")
    ax.scatter(fx, fy, s=6, alpha=0.55, color="tab:red", label="noisy fixes")
    ax.set_aspect("equal")
    ax.set_title(f"$\\sigma$={sigma:.0f} m, {len(fixes['t'])}/{len(truth['t'])} fixes kept")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
axes[0].legend(loc="best", fontsize=9)
fig.suptitle("Clean centreline vs noisy GPS fixes — HSR Layout", fontsize=12)
fig.tight_layout()
plt.show()

## Observation

At σ=15 m the fixes hug the centreline with occasional cross-street excursions; at σ=40 m whole clusters sit a full block off the road — exactly where independent nearest-segment snapping fails and the HMM transition model has to earn its keep. The tunnel gap appears as a long run of missing fixes, so the decoder must bridge it using spatial network consistency.